# Advanced RAG 4단계 파이프라인 — 셀 단위 따라가기

이 노트북은 `rag/` 의 4단계(+0단계)를 **한 셀씩 눈으로 확인**하며 배우는 교재입니다.

```
[0단계] 질의 문맥화   contextualize  (멀티턴: 후속질문 → 독립형)
[1단계] 질의 변환     transform      (hyde / multi_query / decompose)
[2단계] 검색 + 리랭킹  retrieve+rerank (하이브리드 BM25+벡터 → 크로스인코더)
[3단계] 생성          generate       (인용 기반 답변)
[4단계] 검증          verify         (근거 충실성 판정 → 필요시 재생성)
```

- 선행: `python scripts/ingest_sample.py` (또는 arXiv/로컬 수집)로 벡터DB가 있어야 합니다.
- 3·4단계와 질의변환(hyde/multi_query)은 **LLM 호출**이라 `ANTHROPIC_API_KEY` 가 필요합니다.
  2단계(검색·리랭킹)는 로컬 임베딩이라 키 없이도 됩니다.

In [ ]:
# 프로젝트 루트를 import 경로에 추가 (notebooks/ 에서 실행)
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))

from config import settings
print('질의변환 모드:', settings.query_transform_mode, '| 하이브리드:', settings.use_hybrid)
print('리랭크 top_k:', settings.top_k_rerank, '| 검색 top_k:', settings.top_k_retrieve)

## 0단계 — 질의 문맥화 (멀티턴)

대화 히스토리가 있으면 후속 질문("그럼 둘 중 뭐가 더 느린가요?")을 **독립형 질문**으로
재작성합니다. 히스토리가 없으면 원 질문을 그대로 돌려줍니다(무동작).

In [ ]:
from rag.contextualize import contextualize

history = [
    ('user', 'bi-encoder와 cross-encoder의 차이가 뭔가요?'),
    ('assistant', 'bi-encoder는 따로 임베딩해 빠르고, cross-encoder는 함께 보고 정확합니다.'),
]
standalone = contextualize('그럼 둘 중 뭐가 더 느린가요?', history)
print('재작성된 독립형 질문:', standalone)

## 1단계 — 질의 변환

- **hyde**(기본): LLM이 '가설 답변 문서'를 생성 → 그 임베딩으로 검색(문서↔문서 매칭)
- **multi_query**: 질문을 여러 각도로 재작성
- **decompose**: 복합 질문을 하위 질문으로 분해

왜 선택이 hyde인지는 `docs/ADR-001-query-transform-default.md` 참고.

In [ ]:
from rag.stage1_query_transform import transform_query

question = 'REALM과 RETRO는 검색을 사전학습에 통합하는 방식이 어떻게 다른가요?'
queries = transform_query(question, settings.query_transform_mode)
print(f'[{settings.query_transform_mode}] 생성된 검색 질의 {len(queries)}개:')
for i, q in enumerate(queries, 1):
    print(f'  {i}. {q[:120]}')

## 2단계 — 검색 + 리랭킹 (LLM 불필요)

`retrieve_and_rerank` 는 (1차 검색결과, 리랭킹결과)를 함께 돌려줍니다.
1차는 하이브리드(BM25+벡터 RRF), 2차는 크로스인코더(bge-reranker)로 재정렬.
**리랭킹 전후 순위가 어떻게 바뀌는지** 비교해 보세요.

In [ ]:
from rag.vectorstore import load_vectorstore
from rag.stage2_rerank import retrieve_and_rerank

vs = load_vectorstore()
retrieved, reranked = retrieve_and_rerank(vs, question, queries)

print('— 1차 검색 top5 (하이브리드) —')
for c in retrieved[:5]:
    print(f'  [{c.source}] {c.title[:50]} (score={c.score:.3f})')
print('\n— 리랭킹 top5 (크로스인코더) —')
for c in reranked[:5]:
    print(f'  [{c.source}] {c.title[:50]} (rerank={c.rerank_score:.3f} / sec={c.section})')

## 3단계 — 생성 (인용 기반)

리랭킹된 근거를 `[n]` 번호가 달린 컨텍스트로 만들어 LLM이 **출처를 인용하며** 답하게 합니다.

In [ ]:
from rag.stage3_generate import format_context, generate

print('— LLM에 들어가는 컨텍스트(일부) —')
print(format_context(reranked)[:600], '...')

answer = generate(question, reranked)
print('\n— 생성된 답변 —')
print(answer)

## 4단계 — 검증 (LLM-as-judge)

답변의 각 주장이 근거에 실제로 뒷받침되는지(groundedness/faithfulness) 판정합니다.
충실도가 낮으면 파이프라인은 1회 재생성합니다.

In [ ]:
from rag.stage4_verify import verify

v = verify(answer, reranked)
print('grounded     :', v.grounded)
print('faithfulness :', v.faithfulness)
print('미지원 주장   :', v.unsupported_claims)
print('판정 근거     :', v.reasoning[:300])

## 전체 파이프라인 한 번에

위 단계를 `RAGPipeline.run()` 하나로 실행합니다. `.timings` 로 단계별 소요도 볼 수 있습니다.

In [ ]:
from rag.pipeline import RAGPipeline

pipe = RAGPipeline()
result = pipe.run(question)
print('표준화 질문:', result.standalone_question)
print('검색→리랭크:', len(result.retrieved), '→', len(result.reranked))
print('재생성 여부:', result.regenerated)
print('소요(초)   :', {k: round(t, 2) for k, t in (result.timings or {}).items()})
print('\n답변:\n', result.answer)

## 평가 — 무료 검색 지표로 빠르게 점검

`expected_sources`(정답 논문)와 검색 결과만으로 recall@k·MRR 을 계산합니다(LLM 불필요).
평가셋은 `eval/eval_set.jsonl` 한 곳에서 관리합니다. 비싼 LLM 심판(faithfulness 등)은
`python eval/compare.py` 로 설정별 A/B 를 돌려 비교하세요.

In [ ]:
from eval.dataset import load_eval_set
from eval.retrieval_metrics import recall_at_k, reciprocal_rank

k = settings.top_k_rerank
for rec in load_eval_set(difficulty='hard', limit=3):
    _, rr = retrieve_and_rerank(vs, rec['question'], [rec['question']])
    r = recall_at_k(rr, rec['expected_sources'], k)
    m = reciprocal_rank(rr, rec['expected_sources'])
    print(f"{rec['id']}: recall@{k}={r:.2f} mrr={m:.2f}  기대={rec['expected_sources']}")

---
### 더 해보기
- `settings.query_transform_mode` 를 `multi_query`/`decompose` 로 바꿔 1·2단계 결과 비교
- `settings.use_hybrid = False` 로 두고 검색 품질 변화 관찰
- `eval/eval_set.jsonl` 에 질문을 추가해 평가셋 확장
- `python eval/compare.py --retrieval-only` 로 LLM 없이 설정별 검색 품질 스크리닝